## Imports and Setup

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import base64
import mimetypes

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API key exists and begins {openai_api_key[:8]}")
else:
    print("OPENAI_API_KEY not found")
    
client = OpenAI()

OpenAI API key exists and begins sk-proj-


## Model Choice

In [3]:
MODEL_OPTIONS = [
    "gpt-4.1-mini",
    "gpt-4.1",
    "gpt-4o-mini",
]

DEFAULT_MODEL = "gpt-4.1-mini"

system_message = """
You are Study Buddy AI, a patient technical learning assistant.

Your job is to help the user understand programming, AI engineering,
tools, APIs, debugging, and course exercises.

Explain concepts simply, step by step.
Do not dump too much code at once.
When code is needed, explain the idea first, then provide the code.
If the user seems confused, slow down and use examples.
"""

## Helper Function

In [4]:
def image_file_to_data_url(file_path):
    mime_type, _ = mimetypes.guess_type(file_path)
    
    if mime_type is None:
        mime_type = "image/png"
    
    with open(file_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    
    return f"data:{mime_type};base64,{base64_image}"

In [5]:
def convert_gradio_part_to_openai_part(part):
    if part.get("type") == "text":
        return {
            "type": "text",
            "text": part.get("text", "")
        }

    if part.get("type") == "file":
        file_info = part.get("file", {})
        file_path = file_info.get("path")

        if file_path:
            mime_type, _ = mimetypes.guess_type(file_path)

            if mime_type and mime_type.startswith("image/"):
                image_data_url = image_file_to_data_url(file_path)

                return {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data_url
                    }
                }

    return None

In [6]:
def convert_gradio_history_to_openai_history(history):
    openai_history = []

    for message in history:
        role = message.get("role")
        content = message.get("content")

        if role == "assistant":
            openai_history.append({
                "role": "assistant",
                "content": content
            })

        elif role == "user":
            openai_content = []

            if isinstance(content, str):
                openai_content.append({
                    "type": "text",
                    "text": content
                })

            elif isinstance(content, list):
                for part in content:
                    converted_part = convert_gradio_part_to_openai_part(part)

                    if converted_part is not None:
                        openai_content.append(converted_part)

            openai_history.append({
                "role": "user",
                "content": openai_content
            })

    return openai_history

## Chat Function

In [7]:
def chat(message, history, model):
    messages = [{"role": "system", "content": system_message}]

    clean_history = convert_gradio_history_to_openai_history(history)

    for item in clean_history:
        messages.append(item)

    user_content = []

    user_text = message.get("text", "")
    uploaded_files = message.get("files", [])

    if user_text:
        user_content.append({
            "type": "text",
            "text": user_text
        })

    for file_path in uploaded_files:
        mime_type, _ = mimetypes.guess_type(file_path)

        if mime_type and mime_type.startswith("image/"):
            image_data_url = image_file_to_data_url(file_path)

            user_content.append({
                "type": "image_url",
                "image_url": {
                    "url": image_data_url
                }
            })
        else:
            user_content.append({
                "type": "text",
                "text": f"[Uploaded file ignored because it is not an image: {file_path}]"
            })

    messages.append({
        "role": "user",
        "content": user_content
    })
    
    # ------- Check working ------------
    print("MESSAGES SENT TO OPENAI:")
    print(json.dumps(messages, indent=2)[:3000])
    # ------- Check working ------------

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        response += delta
        yield response

In [8]:
with gr.Blocks() as demo:
    gr.Markdown("# Study Buddy AI")
    gr.Markdown("A multimodal learning assistant. For now: text chat + streaming + model switch.")

    model_dropdown = gr.Dropdown(
        choices=MODEL_OPTIONS,
        value=DEFAULT_MODEL,
        label="Choose model"
    )

    gr.ChatInterface(
        fn=chat,
        multimodal=True,
        additional_inputs=[model_dropdown],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/mrinalsood/Developer/forked-repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


MESSAGES SENT TO OPENAI:
[
  {
    "role": "system",
    "content": "\nYou are Study Buddy AI, a patient technical learning assistant.\n\nYour job is to help the user understand programming, AI engineering,\ntools, APIs, debugging, and course exercises.\n\nExplain concepts simply, step by step.\nDo not dump too much code at once.\nWhen code is needed, explain the idea first, then provide the code.\nIf the user seems confused, slow down and use examples.\n"
  },
  {
    "role": "user",
    "content": [
      {
        "type": "text",
        "text": "hi"
      }
    ]
  }
]


Traceback (most recent call last):
  File "/Users/mrinalsood/Developer/forked-repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mrinalsood/Developer/forked-repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mrinalsood/Developer/forked-repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mrinalsood/Developer/forked-repos/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1669, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^